# Paperwise AI — Kaggle GPU test

This notebook runs the existing Paperwise FastAPI grading backend and tests real model inference on a Kaggle GPU. It does **not** deploy a permanent API or change the dashboard.

## Before running
1. In Kaggle, open **Settings → Accelerator → GPU** (if available).
2. Enable Internet in notebook settings so the public model and repository can be downloaded.
3. Run cells from top to bottom. Model download and first load can take several minutes.

Keep the notebook private if you later add private student data. Do not paste API tokens or passwords into notebook cells.

In [ ]:
import os, subprocess, sys, torch
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Switch Kaggle Accelerator to GPU before running model inference.')

In [ ]:
REPO = '/kaggle/working/Paperwise-'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Ayushyadav4579/Paperwise-.git', REPO], check=True)
os.chdir(REPO)
print('Repository:', REPO)
print('Backend exists:', os.path.isfile('server/app.py'))

In [ ]:
# Install app dependencies without forcing a PyTorch reinstall on Kaggle.
%pip install -q 'transformers>=4.45,<5' 'accelerate>=1.0' 'fastapi>=0.115,<1' 'pydantic>=2.8,<3' 'pytest>=8,<9' 'httpx>=0.27,<1'

In [ ]:
# Run the repository's automated API tests first.
os.chdir('/kaggle/working/Paperwise-/server')
test = subprocess.run([sys.executable, '-m', 'pytest', '-q'], text=True, capture_output=True)
print(test.stdout)
print(test.stderr[-4000:] if test.stderr else '')
if test.returncode != 0:
    raise RuntimeError('Automated tests failed. Fix these before evaluating model quality.')
print('AUTOMATED TESTS PASSED')

## Real GPU model inference

The next cell loads the same model configured by the existing API and grades three small sample answers. The output is an experiment, not a guarantee of marking accuracy. Compare every score with a teacher-approved marking scheme before using Paperwise for real exams.

In [ ]:
os.chdir('/kaggle/working/Paperwise-/server')
os.environ.setdefault('PAPERWISE_MODEL_ID', 'Qwen/Qwen2.5-1.5B-Instruct')
from app import GradeRequest, grade, health
print('Health before model load:', health())
sample = GradeRequest(
    board='CBSE', grade='10',
    questions=[
        {'id': 1, 'prompt': 'Why do plants need sunlight?', 'answer': 'Plants use sunlight for photosynthesis to make food.', 'marks': 2, 'criteria': '1 mark: sunlight provides energy; 1 mark: photosynthesis/food production.'},
        {'id': 2, 'prompt': 'State Ohm\'s law.', 'answer': 'At constant temperature, current through a conductor is directly proportional to potential difference across it.', 'marks': 2, 'criteria': '1 mark: current proportional to potential difference; 1 mark: constant physical conditions or V=IR.'},
        {'id': 3, 'prompt': 'What is photosynthesis?', 'answer': 'It is when plants prepare food using light energy, carbon dioxide and water, releasing oxygen.', 'marks': 3, 'criteria': 'Credit food/glucose formation, light/chlorophyll role, carbon dioxide and water as inputs; oxygen is released.'}
    ]
)
result = grade(sample)
import json
print(json.dumps(result, indent=2, ensure_ascii=False))
print('Health after model load:', health())

## What this test proves — and what it does not
- A passing automated test checks the API's validation and response handling.
- The inference cell checks that the selected model can load and return grading results in this session.
- The three sample outputs are not a statistically meaningful accuracy benchmark. Build a larger teacher-scored set before trusting marks.
- Kaggle notebook sessions can stop; this notebook does not create a permanent public API URL. Do not expose an unauthenticated grading endpoint through a public tunnel.